In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [ ]:

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("imdb")

train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

In [ ]:

train_df = train_df.sample(3000, random_state=42)
test_df = test_df.sample(1000, random_state=42)

In [ ]:
# Remove null values
train_df.dropna(inplace=True)

# Lowercase text
train_df['text'] = train_df['text'].str.lower()

In [ ]:
import matplotlib.pyplot as plt

train_df['label'].value_counts().plot(kind='bar')
plt.title("Class Distribution (0 = Negative, 1 = Positive)")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'], train_df['label'], test_size=0.2
)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)

test_encodings = tokenizer(list(test_df['text']), truncation=True, padding=True)

In [ ]:
class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


In [ ]:
def train_model(model, train_loader, optimizer, device):
    model.train()

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

In [ ]:
def evaluate_model(model, val_loader, device):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true.extend(batch['labels'].numpy())

    accuracy = accuracy_score(true, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(true, preds, average='binary')

    return accuracy, precision, recall, f1

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,  # You can adjust this
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
predictions = trainer.predict(val_dataset)



In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

preds = np.argmax(predictions.predictions, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(val_labels, preds, average='binary')
accuracy = accuracy_score(val_labels, preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
from sklearn.metrics import confusion_matrix

predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)

cm = confusion_matrix(val_labels, preds)
print("Confusion Matrix:\n", cm)

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_labels, preds)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(val_labels, preds))

In [ ]:

for param in model.base_model.parameters():
    param.requires_grad = False

In [ ]:

for name, param in model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
models = ['Full BERT', 'Frozen', 'Last 2 Layers']
accuracy = [0.88, 0.75, 0.85]  # replace with your values

plt.bar(models, accuracy)
plt.title("Model Comparison (Accuracy)")
plt.xlabel("Model Type")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df.set_index("Experiment").plot(kind="bar")
plt.title("Model Comparison")
plt.show()

🔹Full Fine-Tuning

Achieves highest accuracy and F1 score

Reason:

All BERT layers are updated

Model fully adapts to dataset

👉 Strongest model, but expensive

🔹 Freeze BERT

Lowest performance

Reason:

BERT acts like a fixed feature extractor

Only classifier layer learns

👉 Fast but weak — limited learning capacity

🔹 Last 2 Layers Fine-Tuning

Performance close to full fine-tuning

Reason:

Higher layers capture task-specific features

Lower layers remain stable

👉 Smart balance between performance and efficiency

Conclusion:

BERT performs well for sentiment classification